<a href="https://colab.research.google.com/github/sukritghosh886-hub/-ai-model-reliability-lab./blob/main/AI_Model_Reliability_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


%pip -q install "fastapi>=0.110,<1.0" "uvicorn>=0.27,<1.0" "scikit-learn>=1.4,<2.0" "httpx>=0.26,<1.0" "pytest>=8,<9"
print("Installation completed")

Installation completed


In [8]:


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Small, controlled test dataset
y_true = [1, 1, 0, 0, 1, 0]
y_pred = [1, 0, 0, 0, 1, 1]

results = {
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1_score": f1_score(y_true, y_pred, zero_division=0),
}

matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])

assert results["accuracy"] == 4 / 6
assert results["precision"] == 2 / 3
assert results["recall"] == 2 / 3
assert results["f1_score"] == 2 / 3
assert matrix.tolist() == [[2, 1], [1, 2]]

print("AI MODEL RELIABILITY LAB — TEST RESULTS")
print("---------------------------------------")
for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

print("\nConfusion matrix (actual rows, predicted columns):")
print(matrix)
print("\nPASS: all expected metric values match.")

AI MODEL RELIABILITY LAB — TEST RESULTS
---------------------------------------
accuracy: 0.6667
precision: 0.6667
recall: 0.6667
f1_score: 0.6667

Confusion matrix (actual rows, predicted columns):
[[2 1]
 [1 2]]

PASS: all expected metric values match.


In [9]:


from pathlib import Path

project = Path("/content/ai-model-reliability-lab")
project.mkdir(exist_ok=True)

html = r'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>AI Model Reliability Lab</title>
<style>
:root{color-scheme:dark;--bg:#101522;--card:#192235;--line:#344159;--text:#f2f5ff;--muted:#aab6cc;--accent:#8ab4ff}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--text);font:16px system-ui,sans-serif}
main{max-width:920px;margin:auto;padding:24px 16px 50px}
header{padding:24px 0}h1{font-size:clamp(28px,6vw,42px);margin:8px 0}
p{color:var(--muted);line-height:1.6}.tag{color:var(--accent);font-weight:700}
.panel,.metric{background:var(--card);border:1px solid var(--line);border-radius:16px;padding:18px;margin:14px 0}
button,.upload{display:inline-block;background:var(--accent);color:#101522;border:0;border-radius:10px;padding:12px 16px;font-weight:700;cursor:pointer;margin:5px 5px 5px 0}
input{max-width:100%;margin:10px 0}button:focus,input:focus{outline:2px solid white;outline-offset:3px}
.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:12px}
.metric small{color:var(--muted)}.value{font-size:28px;font-weight:750;margin-top:8px;overflow-wrap:anywhere}
table{width:100%;border-collapse:collapse;margin-top:12px}th,td{padding:12px;border:1px solid var(--line);text-align:center}
#status{overflow-wrap:anywhere}footer{margin-top:26px;color:var(--muted);font-size:13px}
</style>
</head>
<body><main>
<header>
<div class="tag">OPEN-SOURCE ML EVALUATION TOOL</div>
<h1>AI Model Reliability Lab</h1>
<p>Evaluate binary classification predictions using a small CSV file. No account, GPU, or model download required. Your data is processed in this browser.</p>
</header>
<section class="panel">
<h2>1. Upload your evaluation data</h2>
<p>Required columns: <b>y_true,y_pred</b>. Values must be 0 or 1.
Optional column: <b>latency_ms</b>, a non-negative number for each prediction.</p>
<input id="file" type="file" accept=".csv,text/csv" aria-label="Upload CSV file">
<br><button id="run">Evaluate CSV</button>
<button id="sample">Download sample CSV</button>
<p id="status" role="status">Ready. Upload a CSV or download the sample.</p>
</section>
<section id="results" hidden>
<h2>2. Evaluation results</h2>
<div class="grid">
<div class="metric"><small>Accuracy</small><div class="value" id="accuracy">—</div></div>
<div class="metric"><small>Precision (class 1)</small><div class="value" id="precision">—</div></div>
<div class="metric"><small>Recall (class 1)</small><div class="value" id="recall">—</div></div>
<div class="metric"><small>F1 score (class 1)</small><div class="value" id="f1">—</div></div>
<div class="metric"><small>Evaluated rows</small><div class="value" id="count">—</div></div>
<div class="metric"><small>Mean latency</small><div class="value" id="latency">Not supplied</div></div>
</div>
<div class="panel">
<h3>Confusion matrix</h3>
<p>Rows = actual labels; columns = predicted labels.</p>
<table><thead><tr><th>Actual / Predicted</th><th>0</th><th>1</th></tr></thead>
<tbody><tr><th>0</th><td id="tn">—</td><td id="fp">—</td></tr>
<tr><th>1</th><td id="fn">—</td><td id="tp">—</td></tr></tbody></table>
</div>
<p>These results describe only the uploaded records. They do not guarantee future model performance or production safety.</p>
</section>
<footer>AI Model Reliability Lab · Browser-based evaluation · Version 1.0</footer>
</main>
<script>
const $ = id => document.getElementById(id);
function download(name, text){
 const a=document.createElement("a");
 a.href=URL.createObjectURL(new Blob([text],{type:"text/csv;charset=utf-8"}));
 a.download=name;a.click();URL.revokeObjectURL(a.href);
}
$("sample").onclick=()=>download("sample_predictions.csv",
"y_true,y_pred,latency_ms\\n1,1,12\\n1,0,15\\n0,0,10\\n0,0,11\\n1,1,13\\n0,1,17\\n");
function parseCSV(text){
 const lines=text.replace(/^\\uFEFF/,"").trim().split(/\\r?\\n/);
 if(lines.length<2)throw Error("CSV must contain a header and at least one data row.");
 const split=line=>line.split(",").map(v=>v.trim().replace(/^"|"$/g,""));
 const headers=split(lines[0]).map(h=>h.toLowerCase());
 const ti=headers.indexOf("y_true"),pi=headers.indexOf("y_pred"),li=headers.indexOf("latency_ms");
 if(ti<0||pi<0)throw Error("Missing required y_true or y_pred column.");
 const rows=lines.slice(1).filter(line=>line.trim()).map((line,i)=>{
  const cols=split(line);
  if(cols.length!==headers.length)throw Error("Invalid column count on data row "+(i+2)+".");
  const t=Number(cols[ti]),p=Number(cols[pi]);
  if(![0,1].includes(t)||![0,1].includes(p))
   throw Error("Rows must contain only 0 or 1 labels. Check row "+(i+2)+".");
  let latency=null;
  if(li>=0&&cols[li]!==""){
   latency=Number(cols[li]);
   if(!Number.isFinite(latency)||latency<0)
    throw Error("Invalid latency on row "+(i+2)+".");
  }
  return {t,p,latency};
 });
 if(!rows.length)throw Error("No usable data rows found.");
 return rows;
}
$("run").onclick=async()=>{
 try{
  $("status").textContent="Reading and evaluating CSV…";
  const f=$("file").files[0];
  if(!f)throw Error("Choose a CSV file first.");
  if(f.size>2*1024*1024)throw Error("For this version, use a CSV smaller than 2 MB.");
  const rows=parseCSV(await f.text());
  let tp=0,tn=0,fp=0,fn=0,correct=0;
  for(const r of rows){
   if(r.t===1&&r.p===1)tp++;
   if(r.t===0&&r.p===0)tn++;
   if(r.t===0&&r.p===1)fp++;
   if(r.t===1&&r.p===0)fn++;
   if(r.t===r.p)correct++;
  }
  const div=(a,b)=>b?a/b:0;
  const precision=div(tp,tp+fp),recall=div(tp,tp+fn);
  const f1=div(2*precision*recall,precision+recall);
  const fmt=n=>(100*n).toFixed(2)+"%";
  $("accuracy").textContent=fmt(correct/rows.length);
  $("precision").textContent=fmt(precision);
  $("recall").textContent=fmt(recall);
  $("f1").textContent=fmt(f1);
  $("count").textContent=rows.length;
  $("tn").textContent=tn;$("fp").textContent=fp;
  $("fn").textContent=fn;$("tp").textContent=tp;
  const times=rows.filter(r=>r.latency!==null).map(r=>r.latency);
  $("latency").textContent=times.length?
   (times.reduce((a,b)=>a+b,0)/times.length).toFixed(2)+" ms":"Not supplied";
  $("results").hidden=false;
  $("status").textContent="Evaluation complete. "+rows.length+" rows processed.";
 }catch(e){$("status").textContent="Could not evaluate: "+e.message;}
};
</script></body></html>'''

(project / "index.html").write_text(html, encoding="utf-8")

sample = """y_true,y_pred,latency_ms
1,1,12
1,0,15
0,0,10
0,0,11
1,1,13
0,1,17
"""
(project / "sample_predictions.csv").write_text(sample, encoding="utf-8")

readme = """# AI Model Reliability Lab

A lightweight browser-based binary classification evaluation tool.

## Features
- Accuracy, precision, recall, and F1 score
- Confusion matrix
- Optional mean latency from a supplied `latency_ms` column
- CSV validation and downloadable sample data
- No model downloads or server-side processing

## CSV format
Required columns: `y_true,y_pred`
Optional column: `latency_ms`
Labels must be 0 or 1. Keep CSV files below 2 MB.

## Limitations
This evaluates supplied predictions; it does not train a model, establish causal explanations, or guarantee real-world reliability. Browser timing is not measured as model inference latency: latency is reported only when supplied in the CSV.

## Run locally
Open `index.html` in a modern browser or host it as a static website.
"""
(project / "README.md").write_text(readme, encoding="utf-8")

print("Created project files:")
for path in sorted(project.iterdir()):
    print(path.name, "-", path.stat().st_size, "bytes")
print("\nProject folder:", project)

Created project files:
.gitignore - 40 bytes
README.md - 797 bytes
app.py - 3031 bytes
index.html - 6327 bytes
requirements.txt - 109 bytes
sample_predictions.csv - 67 bytes

Project folder: /content/ai-model-reliability-lab


In [ ]:


import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, brier_score_loss
)

# Load a small built-in benchmark; no manual download
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target,
    test_size=0.25,
    random_state=42,
    stratify=data.target
)

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=500, random_state=42)
)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)
confidence = probabilities.max(axis=1)

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, zero_division=0)
recall = recall_score(y_test, predictions, zero_division=0)
f1 = f1_score(y_test, predictions, zero_division=0)
errors = np.flatnonzero(predictions != y_test)
high_confidence_errors = [
    int(i) for i in errors if confidence[i] >= 0.80
]

print("AI MODEL RELIABILITY LAB")
print("=" * 40)
print("Test samples:", len(y_test))
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print("\nConfusion matrix:")
print(confusion_matrix(y_test, predictions))
print("\nIncorrect predictions:", len(errors))
print("Incorrect predictions with confidence >= 80%:",
      len(high_confidence_errors))
print("Brier score (lower is better):",
      round(brier_score_loss(y_test, probabilities[:, 1]), 4))
print("\nNote: educational benchmark only, not a clinical tool.")

AI MODEL RELIABILITY LAB
Test samples: 143
Accuracy:  0.9860
Precision: 0.9889
Recall:    0.9889
F1-score:  0.9889

Confusion matrix:
[[52  1]
 [ 1 89]]

Incorrect predictions: 2
Incorrect predictions with confidence >= 80%: 1
Brier score (lower is better): 0.0181

Note: educational benchmark only, not a clinical tool.


In [ ]:


recommendations = []

if recall < 0.80:
    recommendations.append(
        "Investigate false negatives and test the decision threshold."
    )

if precision < 0.80:
    recommendations.append(
        "Investigate false positives and improve feature quality."
    )

if accuracy < 0.80:
    recommendations.append(
        "Review data quality and compare with a baseline model."
    )

if len(high_confidence_errors) > 0:
    recommendations.append(
        "Investigate incorrect predictions made with high confidence."
    )

if not recommendations:
    recommendations.append(
        "No basic metric warning triggered. Test on independent data."
    )

print("RELIABILITY REPORT")
print("-" * 40)
for number, recommendation in enumerate(recommendations, start=1):
    print(f"{number}. {recommendation}")

print("\nImportant: high confidence is not automatically calibrated confidence.")

RELIABILITY REPORT
----------------------------------------
1. Investigate incorrect predictions made with high confidence.

Important: high confidence is not automatically calibrated confidence.


In [ ]:


%%writefile app.py
from typing import List, Optional

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)
import numpy as np

app = FastAPI(
    title="AI Model Reliability Lab",
    description="ML metrics, confidence-error analysis, and reliability warnings",
    version="1.0.0"
)

class EvaluationInput(BaseModel):
    true_labels: List[int] = Field(..., min_length=1)
    predicted_labels: List[int] = Field(..., min_length=1)
    confidences: Optional[List[float]] = None

@app.get("/")
def home():
    return {
        "project": "AI Model Reliability Lab",
        "status": "running",
        "documentation": "/docs"
    }

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/evaluate")
def evaluate(data: EvaluationInput):
    true = data.true_labels
    pred = data.predicted_labels
    conf = data.confidences

    if len(true) != len(pred):
        raise HTTPException(422, "Label lists must have equal lengths.")

    if any(x not in (0, 1) for x in true + pred):
        raise HTTPException(422, "Labels must be 0 or 1.")

    if conf is not None:
        if len(conf) != len(true):
            raise HTTPException(422, "Confidence count must match labels.")
        if any(not 0 <= c <= 1 for c in conf):
            raise HTTPException(422, "Confidence must be between 0 and 1.")

    errors = [i for i, (t, p) in enumerate(zip(true, pred)) if t != p]

    result = {
        "sample_count": len(true),
        "accuracy": round(accuracy_score(true, pred), 4),
        "precision": round(precision_score(
            true, pred, zero_division=0
        ), 4),
        "recall": round(recall_score(
            true, pred, zero_division=0
        ), 4),
        "f1_score": round(f1_score(
            true, pred, zero_division=0
        ), 4),
        "confusion_matrix": confusion_matrix(
            true, pred, labels=[0, 1]
        ).tolist(),
        "error_count": len(errors),
        "recommendations": []
    }

    if result["recall"] < 0.8:
        result["recommendations"].append(
            "Review false negatives and decision thresholds."
        )

    if result["precision"] < 0.8:
        result["recommendations"].append(
            "Review false positives and feature quality."
        )

    if conf is not None:
        flagged = [i for i in errors if conf[i] >= 0.8]
        result["confidence_analysis"] = {
            "mean_confidence": round(float(np.mean(conf)), 4),
            "high_confidence_wrong_count": len(flagged),
            "high_confidence_wrong_indices_zero_based": flagged
        }

        if flagged:
            result["recommendations"].append(
                "Review incorrect predictions with confidence >= 80%."
            )

    if not result["recommendations"]:
        result["recommendations"].append(
            "No basic metric warning triggered; test on independent data."
        )

    return result

Writing app.py


In [ ]:


from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

# Test 1: health endpoint
response = client.get("/health")
assert response.status_code == 200
assert response.json()["status"] == "ok"
print("PASS: health check")

# Test 2: metric evaluation
response = client.post("/evaluate", json={
    "true_labels": [0, 1, 1, 0],
    "predicted_labels": [0, 1, 0, 0],
    "confidences": [0.91, 0.95, 0.88, 0.79]
})
assert response.status_code == 200
result = response.json()
assert result["sample_count"] == 4
assert result["error_count"] == 1
assert result["confidence_analysis"]["high_confidence_wrong_count"] == 1
print("PASS: metrics and confidence analysis")
print(result)

# Test 3: reject invalid labels
response = client.post("/evaluate", json={
    "true_labels": [0, 2],
    "predicted_labels": [0, 1]
})
assert response.status_code == 422
print("PASS: input validation")

print("\nAll three API tests passed.")

PASS: health check
PASS: metrics and confidence analysis
{'sample_count': 4, 'accuracy': 0.75, 'precision': 1.0, 'recall': 0.5, 'f1_score': 0.6667, 'confusion_matrix': [[2, 0], [1, 1]], 'error_count': 1, 'recommendations': ['Review false negatives and decision thresholds.', 'Review incorrect predictions with confidence >= 80%.'], 'confidence_analysis': {'mean_confidence': 0.8825, 'high_confidence_wrong_count': 1, 'high_confidence_wrong_indices_zero_based': [2]}}
PASS: input validation

All three API tests passed.


In [ ]:


from pathlib import Path

project = Path("/content/ai-model-reliability-lab")
project.mkdir(exist_ok=True)

(project / "app.py").write_text(
    Path("/content/app.py").read_text()
)

(project / "requirements.txt").write_text(
    "fastapi>=0.110,<1.0\n"
    "uvicorn>=0.27,<1.0\n"
    "scikit-learn>=1.4,<2.0\n"
    "numpy>=1.24,<3.0\n"
    "httpx>=0.26,<1.0\n"
    "pytest>=8,<9\n"
)

(project / "README.md").write_text("""# AI Model Reliability Lab

A lightweight ML evaluation and reliability API.

## Features
- Accuracy, precision, recall, and F1-score
- Confusion matrix and error counting
- High-confidence incorrect prediction flags
- Input validation
- FastAPI interactive documentation
- Automated tests

## Run
pip install -r requirements.txt
uvicorn app:app --host 0.0.0.0 --port 7860

Open /docs for the API interface and /health for the health check.

## Limitations
This is an educational benchmark. Confidence warnings do not prove
probability calibration. Results do not establish real-world
generalization, fairness, or clinical suitability.
""")

(project / ".gitignore").write_text(
    "__pycache__/\n.pytest_cache/\n.env\n.venv/\n"
)

print("Project files:")
for f in sorted(project.iterdir()):
    print(f.name)

Project files:
.gitignore
README.md
app.py
requirements.txt


In [ ]:


import shutil
from google.colab import files

zip_path = shutil.make_archive(
    "/content/ai-model-reliability-lab",
    "zip",
    root_dir="/content/ai-model-reliability-lab"
)

print("ZIP ready:", zip_path)
files.download(zip_path)

ZIP ready: /content/ai-model-reliability-lab.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>